# Run this to enable loading from relative packes

In [1]:
%load_ext autoreload
%autoreload 2
if "PKG" not in globals():
  import importlib, sys, pathlib # https://stackoverflow.com/a/50395128/11996983
  PKG = %pwd
  PKG = pathlib.Path(PKG)
  root_parent_level = 2
  root = PKG
  full_pkg = f"{root.name}"
  for _ in range(root_parent_level):
    root = root.parent
    full_pkg = f"{root.name}.{full_pkg}"
    MODULE_PATH = f"{root}{pathlib.os.path.sep}__init__.py"
    MODULE_NAME = f"{root.name}"
    spec = importlib.util.spec_from_file_location(MODULE_NAME, MODULE_PATH)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
  __package__ = full_pkg


In [2]:
%matplotlib widget
#mpl.rcParams['toolbar'] = 'None

# Save plots with no embeded fonts

In [3]:
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

# Figures Save Path

In [4]:
import platform
import os
if "Ubuntu" in platform.version():
  os.environ["OnedriveConsumer"] = rf"{os.path.expanduser('~')}/OnedrivePersonal/"
  os.environ["OnedriveCommercial"] = rf"{os.path.expanduser('~')}/OnedriveFloating/"

# Common Imports

In [5]:
import numpy as np
import pandas as pd

In [6]:
from .model.util import assignQuantiles
from .model.initvals import NUM_CPUS, InitVals, DT, T_dur
init_vals = InitVals()

# Load all mice data

In [7]:
_FP = "../../paper_fast_slow/data/behavior/df_behavior_full.pkl"
df_behavior = pd.read_pickle(_FP)

In [8]:
def mergeDF(df_full, df_filtered):
    df_filtered_sess = df_filtered[["Name", "Date", "SessionNum"]]
    df_sess__keys = df_filtered_sess.value_counts().index
    df_li = []
    for sess_key in df_sess__keys:
        # print("Sess keys: ", sess_key)
        name, date, sess_num = sess_key
        df_sess = df_full[(df_full["Name"] == name) &
                          (df_full["Date"] == date) &
                          (df_full["SessionNum"] == sess_num)]
        if df_sess.DV.isnull().any():
            print(f"Nan in DV: {df_sess.DV.isnull().sum()}/{len(df_sess)}")
            continue
        df_sess = df_sess.sort_values("TrialNumber")
        df_li.append(df_sess)
    df = pd.concat(df_li).reset_index(drop=True)
    return df

def loadMerged():
    _FP2 = "../../paper_fast_slow/data/behavior/df_behavior.pkl"
    df_behavior = pd.read_pickle(_FP2)
    df_behavior_full = pd.read_pickle(r"/home/main/OnedriveFloatingPersonal/BpodUser/Data/Aggregates/AllAnimals__2024_01_04.pkl")
    df_behavior = mergeDF(df_behavior_full, df_behavior)
    df_behavior.to_pickle(_FP)
    return df_behavior

# df_behavior = loadMerged()

In [9]:
df_ewd = df_behavior[df_behavior.EarlyWithdrawal == 1]
# print(df_ewd.ChoiceCorrect.isnull().sum(), df_ewd.ChoiceLeft.notnull().sum())
df_behavior.loc[df_ewd.index, "ChoiceLeft"] = np.random.choice([0, 1], size=len(df_ewd), p=[0.5, 0.5])
df_behavior.loc[df_ewd.index, "ChoiceCorrect"] = df_behavior.ChoiceLeft == df_behavior.LeftRewarded
df_behavior["ChoiceLeft"] = df_behavior["ChoiceLeft"].astype(float)
df_behavior["ChoiceCorrect"] = df_behavior["ChoiceCorrect"].astype(float)

In [10]:
from ...behavior.util.assigndvstr import assignDVStr
df_behavior = assignDVStr(df_behavior)

In [ ]:
print(f"Nullifying: {(df_behavior.calcStimulusTime > T_dur).sum():,}/{len(df_behavior):,} trial")
df_behavior.loc[df_behavior.calcStimulusTime > T_dur, "ChoiceCorrect"] = np.nan # Treat as no choice
df_behavior.loc[df_behavior.calcStimulusTime > T_dur, "calcStimulusTime"] = np.nan # Treat as no decision time

Nullifying: 4,120/81,080 trial


# Generic function to run model fitting

In [1]:
from .model.initvals import InitVals, NUM_CPUS
from .model.fit import simulateDDM
from .model.drift import DRIFT_FN_DICT
from .model.bias import BIAS_FN_DICT
from .model.noise import _noiseNormal
run_logger = None # Disable logger otherwise we run out of memory


def runModel(drift_fn_str, bias_fn_str, evolvs_res : dict = None,
             num_cpus=NUM_CPUS, dry_run=False):
    noiseFn = _noiseNormal
    driftFn = DRIFT_FN_DICT[drift_fn_str]
    biasFn = BIAS_FN_DICT[bias_fn_str]
    if evolvs_res is None:
        evolvs_res = {}
    
    init_vals = InitVals()
    init_vals_dict = init_vals.toDict()

    optim_result = simulateDDM(df_behavior,
                               driftFn=driftFn, 
                               #include_Q=include_Q, include_RewardRate=include_RewardRate,
                               biasFn=biasFn,
                               noiseFn=noiseFn,
                               bounds_and_defaults=init_vals_dict,
                               dt=DT, t_dur=T_dur,
                               num_cpus=num_cpus,
                               evolvs_res=evolvs_res,
                               dry_run=dry_run)
    return optim_result

for drift_fn_str in DRIFT_FN_DICT:
    for bias_fn_str in BIAS_FN_DICT:
        print("Drift:", drift_fn_str, "Bias:", bias_fn_str)
        runModel(drift_fn_str=drift_fn_str, bias_fn_str=bias_fn_str, num_cpus=1, dry_run=True)
        print()

ImportError: attempted relative import with no known parent package

In [4]:
[print(f'--remove-subject "{s}"', end=' ') for s in 
['Avgat2', 'Avgat3', 'BVGAT1', 'GP4-24', 'GP4-80', 'GP4-81', 'GP4-85', 'RDK_WT1', 'RDK_WT6', 'Rbp4_M2_1', 'WF10', 'WF11', 
 'vgat-40', 'vgat-94', 'vgat-95', 'vgat-96', 'vgat2.5', 'vgatchr2-sk', 'widefield_1']]


--remove-subject "Avgat2" --remove-subject "Avgat3" --remove-subject "BVGAT1" --remove-subject "GP4-24" --remove-subject "GP4-80" --remove-subject "GP4-81" --remove-subject "GP4-85" --remove-subject "RDK_WT1" --remove-subject "RDK_WT6" --remove-subject "Rbp4_M2_1" --remove-subject "WF10" --remove-subject "WF11" --remove-subject "vgat-40" --remove-subject "vgat-94" --remove-subject "vgat-95" --remove-subject "vgat-96" --remove-subject "vgat2.5" --remove-subject "vgatchr2-sk" --remove-subject "widefield_1" 

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [ ]:
from .model.fit import simulateDDM
from .model.drift import DRIFT_FN_DICT
from .model.bias import _biasFixed, BiasAffinity
from .model.noise import _noiseNormal
from inspect import signature
run_logger = None # Disable logger otherwise we run out of memory

def _setFlags(_driftFn):
    drift_fn_parans = signature(_driftFn).parameters
    

def runModel(drift_fn_str, evolvs_res : dict = None):
    assert drift_fn_str in DRIFT_FN_DICT.keys()
    if evolvs_res is None:
        evolvs_res = {}
    init_vals_dict = init_vals.toDict()
    init_vals_dict.pop("Bias_Fixed")
    init_vals_dict.pop("Bias_mu")
    init_vals_dict.pop("Bias_sigma")
    DRIFT_FN_STR = "Classic"
    if DRIFT_FN_DICT not in ["Drift Q-RewardRate"]:
        init_vals_dict.pop("Q_VAL_DECAY_RATE")
        init_vals_dict.pop("Q_VAL_COEF")
    optim_result = simulateDDM(df_behavior,
                            # driftFn=DRIFT_FN_DICT["Drift Q-RewardRate"], include_Q=True, include_RewardRate=True,
                            driftFn=DRIFT_FN_DICT[DRIFT_FN_STR], include_Q=False, include_RewardRate=False,
                            biasFn=_biasFixed,
                            noiseFn=_noiseNormal,
                            bounds_and_defaults=init_vals_dict,
                            dt=DT, t_dur=T_dur,
                            num_cpus=NUM_CPUS,
                            evolvs_res=evolvs_res)
    return optim_result

In [12]:
evolvs_res = {}

In [14]:
# import pickle
# with open("evolvs_res_dump_Classic_biasFixed_3s.pkl", 'rb') as f:
#     evolvs_res = pickle.load(f)

In [13]:
from .model.fit import simulateDDM
from .model.drift import DRIFT_FN_DICT
from .model.bias import _biasFixed, BiasAffinity
from .model.noise import _noiseNormal
run_logger = None # Disable logger otherwise we run out of memory
init_vals_dict = init_vals.toDict()
init_vals_dict.pop("Bias_Fixed")
init_vals_dict.pop("Bias_mu")
init_vals_dict.pop("Bias_sigma")
DRIFT_FN_STR = "Classic"
if DRIFT_FN_DICT not in ["Drift Q-RewardRate"]:
    init_vals_dict.pop("Q_VAL_DECAY_RATE")
    init_vals_dict.pop("Q_VAL_COEF")
optim_result = simulateDDM(df_behavior,
                           # driftFn=DRIFT_FN_DICT["Drift Q-RewardRate"], include_Q=True, include_RewardRate=True,
                           driftFn=DRIFT_FN_DICT[DRIFT_FN_STR], include_Q=False, include_RewardRate=False,
                           biasFn=_biasFixed,
                           noiseFn=_noiseNormal,
                           bounds_and_defaults=init_vals_dict,
                           dt=DT, t_dur=T_dur,
                           num_cpus=NUM_CPUS,
                           evolvs_res=evolvs_res)

df = None
biasFn = <function _biasFixed at 0x7f082919e8e0>
biasFn_kwargs = {'offset': 0, 'bias_affinity': <BiasAffinity.CorrectIncorrect: 1>}
driftFn = <function _driftClassic at 0x7f082919eac0>
driftFn_df_cols = []
driftFn_args = []
noiseFn = <function _noiseNormal at 0x7f082919f7e0>
noiseFn_df_cols = []
include_Q = False
include_RewardRate = False
dt = 0.005
t_dur = 3
Params Names: ['DRIFT_COEF', 'NOISE_SIGMA', 'BOUND', 'BIAS_COEF', 'ALPHA', 'BETA', 'NON_DECISION_TIME']
Params init: [1, 15, 1, 0.95, 0.3, 0.3, 0.3]
Params bounds: [(0, 3), (0, 40), (1, 1), (0, 1), (0, 1), (0, 1), (0, 0.6)]
Skipping: []
Subject: Avgat1


/home/oraby/miniconda3/envs/py311/lib/python3.11/site-packages/scipy/optimize/_differentialevolution.py:488: UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
  with DifferentialEvolutionSolver(func, bounds, args=args,


differential_evolution step 1: f(x)= 1534.1986078886312
differential_evolution step 2: f(x)= 1477.2296983758702
differential_evolution step 3: f(x)= 1471.7577726218096
differential_evolution step 4: f(x)= 1383.2733178654291
differential_evolution step 5: f(x)= 1372.736890951276
differential_evolution step 6: f(x)= 1370.1735498839907
differential_evolution step 7: f(x)= 1370.1735498839907
differential_evolution step 8: f(x)= 1370.1735498839907
differential_evolution step 9: f(x)= 1370.1735498839907
differential_evolution step 10: f(x)= 1348.9614849187935
differential_evolution step 11: f(x)= 1308.5893271461716
differential_evolution step 12: f(x)= 1308.5893271461716
differential_evolution step 13: f(x)= 1308.5893271461716
differential_evolution step 14: f(x)= 1308.5893271461716
differential_evolution step 15: f(x)= 1308.5893271461716
differential_evolution step 16: f(x)= 1308.5893271461716
differential_evolution step 17: f(x)= 1308.5893271461716
differential_evolution step 18: f(x)= 121

In [14]:
evolvs_res = {}

In [ ]:
from .model.fit import simulateDDM
from .model.drift import DRIFT_FN_DICT
from .model.bias import _biasQVal
from .model.noise import _noiseNormal
run_logger = None # Disable logger otherwise we run out of memory
init_vals_dict = init_vals.toDict()
init_vals_dict.pop("Bias_Fixed")
init_vals_dict.pop("Bias_mu")
init_vals_dict.pop("Bias_sigma")
DRIFT_FN_STR = "Classic"
if DRIFT_FN_DICT not in ["Drift Q-RewardRate"]:
    init_vals_dict.pop("Q_VAL_DECAY_RATE")
    init_vals_dict.pop("Q_VAL_COEF")
optim_result = simulateDDM(df_behavior,
                           # driftFn=DRIFT_FN_DICT["Drift Q-RewardRate"], include_Q=True, include_RewardRate=True,
                           driftFn=DRIFT_FN_DICT[DRIFT_FN_STR], include_Q=True, include_RewardRate=False,
                           biasFn=_biasQVal,
                           noiseFn=_noiseNormal,
                           bounds_and_defaults=init_vals_dict,
                           dt=DT, t_dur=T_dur,
                           num_cpus=NUM_CPUS,
                           evolvs_res=evolvs_res)

df = None
biasFn = <function _biasQVal at 0x7f082919e980>
biasFn_kwargs = {}
driftFn = <function _driftClassic at 0x7f082919eac0>
driftFn_df_cols = []
driftFn_args = []
noiseFn = <function _noiseNormal at 0x7f082919f7e0>
noiseFn_df_cols = []
include_Q = True
include_RewardRate = False
dt = 0.005
t_dur = 3
Params Names: ['DRIFT_COEF', 'NOISE_SIGMA', 'BOUND', 'BIAS_COEF', 'ALPHA', 'BETA', 'NON_DECISION_TIME']
Params init: [1, 15, 1, 0.95, 0.3, 0.3, 0.3]
Params bounds: [(0, 3), (0, 40), (1, 1), (0, 1), (0, 1), (0, 1), (0, 0.6)]
Skipping: []
Subject: Avgat1


/home/oraby/miniconda3/envs/py311/lib/python3.11/site-packages/scipy/optimize/_differentialevolution.py:488: UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
  with DifferentialEvolutionSolver(func, bounds, args=args,


differential_evolution step 1: f(x)= 1459.4895591647332
differential_evolution step 2: f(x)= 1459.4895591647332
differential_evolution step 3: f(x)= 1406.924361948956
differential_evolution step 4: f(x)= 1397.6872389791183
differential_evolution step 5: f(x)= 1397.6872389791183
differential_evolution step 6: f(x)= 1390.1299303944315
differential_evolution step 7: f(x)= 1386.4334106728538
differential_evolution step 8: f(x)= 1360.2422273781904
differential_evolution step 9: f(x)= 1356.181902552204
differential_evolution step 10: f(x)= 1298.5976798143852
differential_evolution step 11: f(x)= 1298.5976798143852
differential_evolution step 12: f(x)= 1298.5976798143852
differential_evolution step 13: f(x)= 1298.5976798143852
differential_evolution step 14: f(x)= 1298.5976798143852
differential_evolution step 15: f(x)= 1298.5976798143852
differential_evolution step 16: f(x)= 1298.5976798143852
differential_evolution step 17: f(x)= 1298.5976798143852
differential_evolution step 18: f(x)= 1298

In [ ]:
!mv evolvs_res_dump.pkl evolvs_res_classic_3s_dump.pkl

mv: cannot stat 'evolvs_res_dump.pkl': No such file or directory


In [ ]:
evolvs_res = {}

In [ ]:
from .model.fit import simulateDDM
from .model.drift import DRIFT_FN_DICT
from .model.bias import _biasQVal
from .model.noise import _noiseNormal
run_logger = None # Disable logger otherwise we run out of memory
init_vals_dict = init_vals.toDict()
init_vals_dict.pop("Bias_Fixed")
init_vals_dict.pop("Bias_mu")
init_vals_dict.pop("Bias_sigma")
DRIFT_FN_STR = "Classic+Q"
if DRIFT_FN_STR not in ["Drift Q-RewardRate"]:
    init_vals_dict.pop("Q_VAL_DECAY_RATE")
if DRIFT_FN_STR not in ["Drift Q-RewardRate", "Classic+Q"]:
    init_vals_dict.pop("Q_VAL_COEF")
optim_result = simulateDDM(df_behavior,
                           # driftFn=DRIFT_FN_DICT["Drift Q-RewardRate"], include_Q=True, include_RewardRate=True,
                           driftFn=DRIFT_FN_DICT[DRIFT_FN_STR], include_Q=True, include_RewardRate=True,
                           biasFn=_biasQVal,
                           noiseFn=_noiseNormal,
                           bounds_and_defaults=init_vals_dict,
                           dt=DT, t_dur=T_dur,
                           num_cpus=NUM_CPUS,
                           evolvs_res=evolvs_res)

Fixed Params Names: ['df', 'biasFn', 'biasFn_kwargs', 'driftFn', 'driftFn_df_cols', 'driftFn_args', 'noiseFn', 'noiseFn_df_cols', 'include_Q', 'include_RewardRate', 'dt', 't_dur']
Fixed Params Vals: [None, <function _biasQVal at 0x7f622c9da520>, {}, <function _driftClassicAddQ at 0x7f622c9daa20>, ['RewardRate', 'Q_val'], ['Q_VAL_COEF'], <function _noiseNormal at 0x7f622c9db600>, [], True, True, 0.005, 3]
Params Names: ['DRIFT_COEF', 'NOISE_SIGMA', 'BOUND', 'BIAS_COEF', 'ALPHA', 'BETA', 'NON_DECISION_TIME', 'Q_VAL_COEF']
Params init: [1, 15, 1, 0.95, 0.3, 0.3, 0.3, 5]
Params bounds: [(0, 3), (0, 40), (1, 1), (0, 1), (0, 1), (0, 1), (0, 0.6), (1, 20)]
Skipping: []
Subject: RDK_WT6


/home/oraby/miniconda3/envs/py311/lib/python3.11/site-packages/scipy/optimize/_differentialevolution.py:488: UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
  with DifferentialEvolutionSolver(func, bounds, args=args,


differential_evolution step 1: f(x)= 1157.136086662153
differential_evolution step 2: f(x)= 1157.136086662153
differential_evolution step 3: f(x)= 1157.136086662153
differential_evolution step 4: f(x)= 1157.136086662153
differential_evolution step 5: f(x)= 1154.162491536899
differential_evolution step 6: f(x)= 1145.1618144888287
differential_evolution step 7: f(x)= 1145.1618144888287
differential_evolution step 8: f(x)= 1140.093432633717
differential_evolution step 9: f(x)= 1135.9566689234935
differential_evolution step 10: f(x)= 1135.9566689234935
differential_evolution step 11: f(x)= 1135.9566689234935
differential_evolution step 12: f(x)= 1134.6960054163847
differential_evolution step 13: f(x)= 1094.7921462423833
differential_evolution step 14: f(x)= 1094.7921462423833
differential_evolution step 15: f(x)= 1094.7921462423833
differential_evolution step 16: f(x)= 1094.7921462423833
differential_evolution step 17: f(x)= 1094.7921462423833
differential_evolution step 18: f(x)= 1094.792

In [ ]:
evolvs_res = {}

In [ ]:
from .model.fit import simulateDDM
from .model.drift import DRIFT_FN_DICT
from .model.bias import _biasQVal
from .model.noise import _noiseNormal
run_logger = None # Disable logger otherwise we run out of memory
init_vals_dict = init_vals.toDict()
init_vals_dict.pop("Bias_Fixed")
init_vals_dict.pop("Bias_mu")
init_vals_dict.pop("Bias_sigma")
DRIFT_FN_STR = "Drift Q-RewardRate"
if DRIFT_FN_STR not in ["Drift Q-RewardRate"]:
    init_vals_dict.pop("Q_VAL_DECAY_RATE")
if DRIFT_FN_STR not in ["Drift Q-RewardRate", "Classic+Q"]:
    init_vals_dict.pop("Q_VAL_COEF")
optim_result = simulateDDM(df_behavior,
                           # driftFn=DRIFT_FN_DICT["Drift Q-RewardRate"], include_Q=True, include_RewardRate=True,
                           driftFn=DRIFT_FN_DICT[DRIFT_FN_STR], include_Q=True, include_RewardRate=True,
                           biasFn=_biasQVal,
                           noiseFn=_noiseNormal,
                           bounds_and_defaults=init_vals_dict,
                           dt=DT, t_dur=T_dur,
                           num_cpus=NUM_CPUS,
                           evolvs_res=evolvs_res)

Fixed Params Names: ['df', 'biasFn', 'biasFn_kwargs', 'driftFn', 'driftFn_df_cols', 'driftFn_args', 'noiseFn', 'noiseFn_df_cols', 'include_Q', 'include_RewardRate', 'dt', 't_dur']
Fixed Params Vals: [None, <function _biasQVal at 0x7f622c9da520>, {}, <function _driftDriftQRewardRate at 0x7f62178fd260>, ['RewardRate', 'Q_val'], ['Q_VAL_COEF', 'Q_VAL_DECAY_RATE'], <function _noiseNormal at 0x7f622c9db600>, [], True, True, 0.005, 3]
Params Names: ['DRIFT_COEF', 'NOISE_SIGMA', 'BOUND', 'BIAS_COEF', 'ALPHA', 'BETA', 'NON_DECISION_TIME', 'Q_VAL_DECAY_RATE', 'Q_VAL_COEF']
Params init: [1, 15, 1, 0.95, 0.3, 0.3, 0.3, 1, 5]
Params bounds: [(0, 3), (0, 40), (1, 1), (0, 1), (0, 1), (0, 1), (0, 0.6), (0.1, 4), (1, 20)]
Skipping: []
Subject: RDK_WT6


/home/oraby/miniconda3/envs/py311/lib/python3.11/site-packages/scipy/optimize/_differentialevolution.py:488: UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
  with DifferentialEvolutionSolver(func, bounds, args=args,


differential_evolution step 1: f(x)= 1176.6404874746106
differential_evolution step 2: f(x)= 1167.7251184834122
differential_evolution step 3: f(x)= 1145.9011509817196
differential_evolution step 4: f(x)= 1126.9681787406905
differential_evolution step 5: f(x)= 1073.0792146242384
differential_evolution step 6: f(x)= 1073.0792146242384
differential_evolution step 7: f(x)= 1073.0792146242384
differential_evolution step 8: f(x)= 1073.0792146242384
differential_evolution step 9: f(x)= 1073.0792146242384
differential_evolution step 10: f(x)= 1073.0792146242384
differential_evolution step 11: f(x)= 1073.0792146242384
differential_evolution step 12: f(x)= 1073.0792146242384
differential_evolution step 13: f(x)= 1073.0792146242384
differential_evolution step 14: f(x)= 1073.0792146242384
differential_evolution step 15: f(x)= 1073.0792146242384
differential_evolution step 16: f(x)= 1073.0792146242384
differential_evolution step 17: f(x)= 1073.0792146242384
differential_evolution step 18: f(x)= 10

In [ ]:
evolvs_res = {}

In [ ]:
from .model.fit import simulateDDM
from .model.drift import DRIFT_FN_DICT
from .model.bias import _biasQVal
from .model.noise import _noiseNormal
run_logger = None # Disable logger otherwise we run out of memory
init_vals_dict = init_vals.toDict()
init_vals_dict.pop("Bias_Fixed")
init_vals_dict.pop("Bias_mu")
init_vals_dict.pop("Bias_sigma")
DRIFT_FN_STR = "DriftGain"
if DRIFT_FN_STR not in ["Drift Q-RewardRate"]:
    init_vals_dict.pop("Q_VAL_DECAY_RATE")
if DRIFT_FN_STR not in ["Drift Q-RewardRate", "Classic+Q"]:
    init_vals_dict.pop("Q_VAL_COEF")
optim_result = simulateDDM(df_behavior,
                           # driftFn=DRIFT_FN_DICT["Drift Q-RewardRate"], include_Q=True, include_RewardRate=True,
                           driftFn=DRIFT_FN_DICT[DRIFT_FN_STR], include_Q=True, include_RewardRate=True,
                           biasFn=_biasQVal,
                           noiseFn=_noiseNormal,
                           bounds_and_defaults=init_vals_dict,
                           dt=DT, t_dur=T_dur,
                           num_cpus=NUM_CPUS,
                           evolvs_res=evolvs_res)

Fixed Params Names: ['df', 'biasFn', 'biasFn_kwargs', 'driftFn', 'driftFn_df_cols', 'driftFn_args', 'noiseFn', 'noiseFn_df_cols', 'include_Q', 'include_RewardRate', 'dt', 't_dur']
Fixed Params Vals: [None, <function _biasQVal at 0x7f622c9da520>, {}, <function _driftRewardRateDriftGain at 0x7f62178fcfe0>, ['RewardRate'], [], <function _noiseNormal at 0x7f622c9db600>, [], True, True, 0.005, 3]
Params Names: ['DRIFT_COEF', 'NOISE_SIGMA', 'BOUND', 'BIAS_COEF', 'ALPHA', 'BETA', 'NON_DECISION_TIME']
Params init: [1, 15, 1, 0.95, 0.3, 0.3, 0.3]
Params bounds: [(0, 3), (0, 40), (1, 1), (0, 1), (0, 1), (0, 1), (0, 0.6)]
Skipping: []
Subject: RDK_WT6


/home/oraby/miniconda3/envs/py311/lib/python3.11/site-packages/scipy/optimize/_differentialevolution.py:488: UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
  with DifferentialEvolutionSolver(func, bounds, args=args,


differential_evolution step 1: f(x)= 1185.3419092755585
differential_evolution step 2: f(x)= 1127.5314827352743
differential_evolution step 3: f(x)= 1127.5314827352743
differential_evolution step 4: f(x)= 1127.5314827352743
differential_evolution step 5: f(x)= 1127.5314827352743
differential_evolution step 6: f(x)= 1127.5314827352743
differential_evolution step 7: f(x)= 1127.5314827352743
differential_evolution step 8: f(x)= 1089.462423832092
differential_evolution step 9: f(x)= 1089.462423832092
differential_evolution step 10: f(x)= 1089.462423832092
differential_evolution step 11: f(x)= 1089.462423832092
differential_evolution step 12: f(x)= 1089.462423832092
differential_evolution step 13: f(x)= 1089.462423832092
differential_evolution step 14: f(x)= 1089.462423832092
differential_evolution step 15: f(x)= 1089.462423832092
differential_evolution step 16: f(x)= 1089.462423832092
differential_evolution step 17: f(x)= 1089.462423832092
differential_evolution step 18: f(x)= 1089.4624238

In [ ]:
evolvs_res = {}

In [ ]:
from .model.fit import simulateDDM
from .model.drift import DRIFT_FN_DICT
from .model.bias import _biasQVal
from .model.noise import _noiseNormal
run_logger = None # Disable logger otherwise we run out of memory
init_vals_dict = init_vals.toDict()
init_vals_dict.pop("Bias_Fixed")
init_vals_dict.pop("Bias_mu")
init_vals_dict.pop("Bias_sigma")
DRIFT_FN_STR = "NoiseGain"
if DRIFT_FN_STR not in ["Drift Q-RewardRate"]:
    init_vals_dict.pop("Q_VAL_DECAY_RATE")
if DRIFT_FN_STR not in ["Drift Q-RewardRate", "Classic+Q"]:
    init_vals_dict.pop("Q_VAL_COEF")
optim_result = simulateDDM(df_behavior,
                           # driftFn=DRIFT_FN_DICT["Drift Q-RewardRate"], include_Q=True, include_RewardRate=True,
                           driftFn=DRIFT_FN_DICT[DRIFT_FN_STR], include_Q=True, include_RewardRate=True,
                           biasFn=_biasQVal,
                           noiseFn=_noiseNormal,
                           bounds_and_defaults=init_vals_dict,
                           dt=DT, t_dur=T_dur,
                           num_cpus=NUM_CPUS,
                           evolvs_res=evolvs_res)

Fixed Params Names: ['df', 'biasFn', 'biasFn_kwargs', 'driftFn', 'driftFn_df_cols', 'driftFn_args', 'noiseFn', 'noiseFn_df_cols', 'include_Q', 'include_RewardRate', 'dt', 't_dur']
Fixed Params Vals: [None, <function _biasQVal at 0x7f622c9da520>, {}, <function _driftRewardRateNoiseGain at 0x7f62178fc9a0>, ['RewardRate'], [], <function _noiseNormal at 0x7f622c9db600>, [], True, True, 0.005, 3]
Params Names: ['DRIFT_COEF', 'NOISE_SIGMA', 'BOUND', 'BIAS_COEF', 'ALPHA', 'BETA', 'NON_DECISION_TIME']
Params init: [1, 15, 1, 0.95, 0.3, 0.3, 0.3]
Params bounds: [(0, 3), (0, 40), (1, 1), (0, 1), (0, 1), (0, 1), (0, 0.6)]
Skipping: []
Subject: RDK_WT6


/home/oraby/miniconda3/envs/py311/lib/python3.11/site-packages/scipy/optimize/_differentialevolution.py:488: UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
  with DifferentialEvolutionSolver(func, bounds, args=args,


differential_evolution step 1: f(x)= 1166.6702775897088
differential_evolution step 2: f(x)= 1166.6283006093431
differential_evolution step 3: f(x)= 1166.6283006093431
differential_evolution step 4: f(x)= 1166.6283006093431
differential_evolution step 5: f(x)= 1164.6824644549763
differential_evolution step 6: f(x)= 1164.6824644549763
differential_evolution step 7: f(x)= 1164.6824644549763
differential_evolution step 8: f(x)= 1137.9702098849018
differential_evolution step 9: f(x)= 1136.6418415707515
differential_evolution step 10: f(x)= 1136.6418415707515
differential_evolution step 11: f(x)= 1136.6418415707515
differential_evolution step 12: f(x)= 1136.6418415707515
differential_evolution step 13: f(x)= 1136.6418415707515
differential_evolution step 14: f(x)= 1136.6418415707515
differential_evolution step 15: f(x)= 1136.6418415707515
differential_evolution step 16: f(x)= 1136.6418415707515
differential_evolution step 17: f(x)= 1136.6418415707515
differential_evolution step 18: f(x)= 11